# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanay1122/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

The vector is rebuilt from the warehouse — the same slice as my ML-04 companion notebook (`w03_data_contract`):
Lane 2 (Refresh / Content Opportunity Scoring), month 2026-03 of the release, one row = one page, decision
moment 2026-03-20. Policy restated so this notebook stands alone:

- row-days only where `gsc_data_available IS TRUE` (the release zero-fills windows it cannot see; those fake zeros must never become features);
- feature window 03-01..20, label window 03-21..31; label `decline_11d` = TRUE when the target-window mean daily impressions fall below 70% of the 03-14..20 pace;
- aggregation floor: `SUM(impressions) >= 100` AND `COUNT(*) >= 10` — a written policy, so tiny pages cannot vote;
- five numeric features (table in section 2), no categoricals this week — categorical handling joins the vector in the modeling weeks;
- dropped and counted, never silently: pages with no target-window rows (absence, not zero) and pages whose 03-14..20 pace is 0/NULL (the decline ratio would be unanswerable);
- `momentum` can still be NULL (baseline days all-0/NULL) — left NULL in the frame; the repo's own rank scaler (`scripts/ml_utils.percentile_rank`) maps NULL to the bottom bucket. Declared policy choice, not nature.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Library pinning, token resolution (never typed in a cell), and local access to the release.
%pip -q install duckdb huggingface_hub scikit-learn
import os
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

from pathlib import Path
import subprocess
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm warns when jupyter widgets are absent

import duckdb

print("duckdb", duckdb.__version__)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError(
        "No HF_TOKEN found. One-time setup: (1) accept the gate at "
        "huggingface.co/datasets/FlyRank/internship-warehouse (affiliation 'FlyRank ML Internship 2026'); "
        "(2) create a READ token (Settings -> Access Tokens, type Read); "
        "(3) make it available: locally run  setx HF_TOKEN \"hf_...\"  in a fresh terminal, "
        "or in Colab add a Secret named HF_TOKEN. Then re-run this notebook."
    )
print("token found: present in environment (value not printed)")

from huggingface_hub import snapshot_download

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    # Cloud-runtime fallback: the Colab badge opens just the notebook, not the repo.
    for parent in [Path("/content"), Path.cwd()]:
        target = parent / "flyrank-ml"
        if not target.parent.exists():
            continue
        if not (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            print(f"repo not found locally; cloning the public repo into {target} ...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tanay1122/flyrank-ml.git", str(target)], check=False, capture_output=True)
        if (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return target
    raise FileNotFoundError("No flyrank-ml repo found — run from inside the repo (or let the Colab fallback clone it).")

REPO = find_repo_root()
CACHE = REPO / "work" / "cache" / "warehouse"
LOCAL = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns=["dim_clients.parquet", "fact_content_daily_performance/month=2026-03/*.parquet"],
    token=HF_TOKEN,
    local_dir=str(CACHE),
)
CACHE_DIR = Path(LOCAL)

DIM_CLIENTS = (CACHE_DIR / "dim_clients.parquet").as_posix()
FACT_MAR = (CACHE_DIR / "fact_content_daily_performance" / "month=2026-03" / "data_0.parquet").as_posix()
print("local files ready:")
print("  ", Path(FACT_MAR).name, "-", round(Path(FACT_MAR).stat().st_size / 1e6, 1), "MB (March partition)")
print("  ", Path(DIM_CLIENTS).name)

con = duckdb.connect()
dim_check = con.execute(f"SELECT COUNT(*) AS clients FROM read_parquet('{DIM_CLIENTS}')").df()
print(dim_check.to_string(index=False))


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
duckdb 2.0.0
token found: present in environment (value not printed)


local files ready:
   data_0.parquet - 124.2 MB (March partition)
   dim_clients.parquet


 clients
     104


The code cell builds the eval-ready vector: per-page aggregates over 03-01..20 from real (flag-TRUE) row-days
only, the target-window pace from 03-21..31, the label, and the honest drop ledger. The two drop reasons
are a population definition — audited for outcome-window information in section 3, not hidden.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Build the eval-ready vector: per-page features (flag-TRUE row-days, 03-01..20) + label from 03-21..31.
import pandas as pd

features = con.execute(f'''
    WITH early AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{FACT_MAR}')
        WHERE report_date <= DATE '2026-03-20' AND gsc_data_available IS TRUE
    ), aggregated AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_early_tot,
               SUM(gsc_clicks) AS clicks_early_tot,
               AVG(CASE WHEN report_date >= DATE '2026-03-14' THEN gsc_impressions END) AS pace_mean,
               AVG(CASE WHEN report_date <= DATE '2026-03-13' THEN gsc_impressions END) AS base_mean,
               AVG(gsc_avg_position) AS pos_mean,
               SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_active
        FROM early
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND COUNT(*) >= 10
    )
    SELECT *,
           pace_mean / NULLIF(base_mean, 0) AS momentum,
           100.0 * clicks_early_tot / imp_early_tot AS click_rate_pct
    FROM aggregated
''').df()

target = con.execute(f'''
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_impressions) AS target_mean
    FROM read_parquet('{FACT_MAR}')
    WHERE report_date >= DATE '2026-03-21' AND report_date <= DATE '2026-03-31'
          AND gsc_data_available IS TRUE
    GROUP BY 1, 2
''').df()

frame = features.merge(target, on=["client_hash_id", "content_hash_id"], how="left")
no_outcome = int(frame["target_mean"].isna().sum())
frame = frame[frame["target_mean"].notna()].copy()
pace_ok = frame["pace_mean"] > 0
no_pace = int((~pace_ok).sum())
frame = frame[pace_ok].copy()
frame["decline_11d"] = (frame["target_mean"] < 0.70 * frame["pace_mean"]).astype(int)

feature_cols = ["imp_early_tot", "click_rate_pct", "pos_mean", "days_active", "momentum"]
one_row_one_page = frame["content_hash_id"].is_unique
print(f"Vector built: {len(frame):,} pages x 5 features + label | one row = one page: {one_row_one_page}")
print("Drop ledger (population definition; audited in section 3):")
print(f"  no rows at all in the target window (absent, not zero): {no_outcome:,}")
print(f"  no 03-14..20 baseline pace (0 or NULL): {no_pace:,}")
print(f"Base rate: {frame['decline_11d'].mean() * 100:.1f}% ({int(frame['decline_11d'].sum()):,} of {len(frame):,} pages decline)")
print("Missing values per column (kept rows):")
print(frame[feature_cols + ["decline_11d"]].isna().sum().to_string())

Vector built: 83,258 pages x 5 features + label | one row = one page: True
Drop ledger (population definition; audited in section 3):
  no rows at all in the target window (absent, not zero): 314
  no 03-14..20 baseline pace (0 or NULL): 37
Base rate: 15.9% (13,277 of 83,258 pages decline)
Missing values per column (kept rows):
imp_early_tot     0
click_rate_pct    0
pos_mean          0
days_active       0
momentum          0
decline_11d       0


## 2. Feature notes (meaning, missing, categorical, available-when?)

| # | Feature | Meaning | Missing handling | Knowable BEFORE 2026-03-20? |
|---|---|---|---|---|
| 1 | `imp_early_tot` | total search demand captured so far | none possible (the floor guarantees totals exist) | yes — SUM over flag-TRUE row-days ≤ 03-20 |
| 2 | `click_rate_pct` | share of demand captured by clicks | none | yes — clicks and impressions both from ≤ 03-20 |
| 3 | `pos_mean` | average SERP position so far (lower = better) | none on kept rows (positional NULLs live only on flag-FALSE rows, already filtered) | yes — AVG over row-days ≤ 03-20 |
| 4 | `days_active` | presence/regularity | none | yes — COUNT of row-days ≤ 03-20 |
| 5 | `momentum` | recent pace (03-14..20) vs early pace (01-13) | NULL possible → bottom bucket by the declared rank policy | yes — the latest day it reads IS the decision date |

Label `decline_11d` — target-window only, never a feature; `client_hash_id` / `content_hash_id` — context, grouping, splits only; availability flags — filters, never learned.

One grey zone declared up front: `momentum` shares its 03-14..20 part with the label's baseline window. Legal by the
timeline (it never touches 03-21..31), but exactly the kind of sibling section 3 must test: train with it, without
it, and report both.

**Available-when, tested rather than asserted:** the audit below rebuilds the aggregates twice — the full feature
window (≤ 03-20) and again with the boundary day removed (≤ 03-19) — and compares. If the vector barely moves
without the decision day itself, it encodes history up to the decision moment, not a one-day razor.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Available-when audit: 03-20-capped aggregates vs 03-19-capped aggregates, feature by feature.
import numpy as np

def aggregates(cutoff: str) -> pd.DataFrame:
    return con.execute(f'''
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_early_tot,
               100.0 * SUM(gsc_clicks) / SUM(gsc_impressions) AS click_rate_pct,
               AVG(gsc_avg_position) AS pos_mean,
               COUNT(*) AS days_active,
               AVG(CASE WHEN report_date >= DATE '2026-03-14' THEN gsc_impressions END) AS pace_part,
               AVG(CASE WHEN report_date <= DATE '2026-03-13' THEN gsc_impressions END) AS base_part
        FROM read_parquet('{FACT_MAR}')
        WHERE report_date <= DATE '{cutoff}' AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ''').df()

full = aggregates("2026-03-20")
censored = aggregates("2026-03-19")
joined = full.merge(censored, on=["client_hash_id", "content_hash_id"], suffixes=("_20", "_19"), how="inner")

for col in ["imp_early_tot", "click_rate_pct", "pos_mean", "days_active"]:
    corr = joined[f"{col}_20"].corr(joined[f"{col}_19"])
    print(f"  .corr() {col}: full-window vs boundary-day-removed = {corr:.4f}")

mom_20 = joined["pace_part_20"] / joined["base_part_20"].where(joined["base_part_20"] > 0)
mom_19 = joined["pace_part_19"] / joined["base_part_19"].where(joined["base_part_19"] > 0)
finite = mom_20.notna() & mom_19.notna()
print(f"  .corr() momentum: {mom_20[finite].corr(mom_19[finite]):.4f} (on {int(finite.sum()):,} defined pairs)")
print("Reading: every feature is nearly the same without the boundary day — the vector encodes history up to")
print("the decision moment, not a single final day. (Not 1.0 exactly: one day of impressions DOES shift a few pages.)")

  .corr() imp_early_tot: full-window vs boundary-day-removed = 0.9996
  .corr() click_rate_pct: full-window vs boundary-day-removed = 0.9864
  .corr() pos_mean: full-window vs boundary-day-removed = 0.9955
  .corr() days_active: full-window vs boundary-day-removed = 0.9990
  .corr() momentum: 0.9710 (on 132,768 defined pairs)
Reading: every feature is nearly the same without the boundary day — the vector encodes history up to
the decision moment, not a single final day. (Not 1.0 exactly: one day of impressions DOES shift a few pages.)


## 3. The leakage hunt

Attack receipts for each taxonomy bucket, on this vector:

1. **Label-derived / grey sibling — momentum.** The label's rule is `target_mean < 0.70 x pace_mean`; `momentum` = recent pace / early pace shares the pace term. Train and score WITH it, WITHOUT it; the gap is the confession (if any).
2. **Label-derived, the deliberate trap** — `trap_target_ratio` = pace_mean / target_mean (decline risk born from the answer's own denominator; by definition TRUE iff > ~1/0.7). Added on purpose: quick score jumps toward perfect; then deleted; honest number kept.
3. **Future/overlapping windows** — drawn in the contract: features ≤ 03-20, label ≥ 03-21. Section 2's audit (features stable without the boundary day) plus the trap's perfect score make the window rule concrete: the ONLY column that can see past 03-20 is the trap.
4. **Decision-derived product flags** — not shipped in the release at all (receipt printed in section 4); a lane baseline to beat lives in the baseline weeks, never as input.
5. **Split honesty** — pages from one client repeat across rows and share hidden character: grouped CV by `client_hash_id` is the honest read; a random row split is printed as the memorization contrast; every model metric below is out-of-fold.
6. **Base rate next to every score** — printed in every cell.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Hunt, quick-score level: single-feature AUCs, momentum in/out, the trap in/out (then deleted).
import sys

sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import percentile_rank, precision_at_k

def auc(truth: pd.Series, scores: pd.Series) -> float:
    ranks = scores.rank()
    n_pos = int((truth == 1).sum())
    n_neg = int((truth == 0).sum())
    return float((ranks[truth == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

truth = frame["decline_11d"]
for col in feature_cols:
    print(f"  single-feature AUC {col}: {auc(truth, frame[col].fillna(0)):.3f}")

scaled = pd.DataFrame({col: percentile_rank(frame[col]) for col in feature_cols})
quick_honest = auc(truth, scaled[feature_cols].mean(axis=1))
quick_no_momentum = auc(truth, scaled[[c for c in feature_cols if c != "momentum"]].mean(axis=1))
scaled_trap = scaled.copy()
scaled_trap["trap_target_ratio"] = percentile_rank(frame["pace_mean"] / frame["target_mean"])
quick_with_trap = auc(truth, scaled_trap.mean(axis=1))
quick_trap_alone = auc(truth, frame["pace_mean"] / frame["target_mean"])
del scaled_trap["trap_target_ratio"]

print(f"Quick score (rank-averaged 5 features):            {quick_honest:.3f}")
print(f"Quick score WITHOUT momentum (grey sibling):       {quick_no_momentum:.3f}  (momentum's share of the ranking)")
print(f"Quick score WITH the trap column:                  {quick_with_trap:.3f}  <- jumps toward perfect, by construction")
print(f"Trap column alone:                                 {quick_trap_alone:.3f}  <- it IS the label's own rule")
print(f"Base rate for every score above: {truth.mean() * 100:.1f}% declines; coin-flip ordering = 0.500.")
print("Trap column DELETED. The kept numbers are the honest ones (re-scored at model level below).")

  single-feature AUC imp_early_tot: 0.392
  single-feature AUC click_rate_pct: 0.407
  single-feature AUC pos_mean: 0.520
  single-feature AUC days_active: 0.477
  single-feature AUC momentum: 0.585


Quick score (rank-averaged 5 features):            0.442
Quick score WITHOUT momentum (grey sibling):       0.390  (momentum's share of the ranking)
Quick score WITH the trap column:                  0.665  <- jumps toward perfect, by construction
Trap column alone:                                 1.000  <- it IS the label's own rule
Base rate for every score above: 15.9% declines; coin-flip ordering = 0.500.
Trap column DELETED. The kept numbers are the honest ones (re-scored at model level below).


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Hunt, model level: grouped CV by client (out-of-fold) vs random row split; the trap WITH/WITHOUT; coefficients eyeballed.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict, train_test_split

X = scaled[feature_cols].to_numpy()
y = truth.to_numpy()
groups = frame["client_hash_id"].to_numpy()

oof_honest = cross_val_predict(
    LogisticRegression(max_iter=2000), X, y,
    cv=GroupKFold(n_splits=5), groups=groups, method="predict_proba",
)[:, 1]
auc_grouped = auc(truth, pd.Series(oof_honest, index=frame.index))

X_trap = np.c_[X, percentile_rank(frame["pace_mean"] / frame["target_mean"]).to_numpy()]
oof_trap = cross_val_predict(
    LogisticRegression(max_iter=2000), X_trap, y,
    cv=GroupKFold(n_splits=5), groups=groups, method="predict_proba",
)[:, 1]
X_no_momentum = scaled[[c for c in feature_cols if c != "momentum"]].to_numpy()
oof_no_momentum = cross_val_predict(
    LogisticRegression(max_iter=2000), X_no_momentum, y,
    cv=GroupKFold(n_splits=5), groups=groups, method="predict_proba",
)[:, 1]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)
auc_random = auc(pd.Series(yte), pd.Series(LogisticRegression(max_iter=2000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]))

coefs_honest = LogisticRegression(max_iter=2000).fit(X, y).coef_[0]  # in-sample eyeball of importance, NOT a score
coefs_trap = LogisticRegression(max_iter=2000).fit(X_trap, y).coef_[0]
top = sorted(zip(feature_cols, coefs_honest), key=lambda pair: -abs(pair[1]))[:3]
print(f"Base rate: {y.mean() * 100:.1f}% | coin flip = 0.500 AUC")
print(f"Grouped CV AUC (client-holdout, out-of-fold):  {auc_grouped:.3f}  | precision@20 on OOF scores: {precision_at_k(y, oof_honest, 20):.2f}")
print(f"Random row split AUC (25% test, stratified):   {auc_random:.3f}")
print(f"  memorization receipt = random - grouped = {auc_random - auc_grouped:+.3f}")
print(f"Grouped CV AUC with momentum removed:          {auc(truth, pd.Series(oof_no_momentum, index=frame.index)):.3f}")
print(f"Grouped CV AUC with the trap column added:     {auc(truth, pd.Series(oof_trap, index=frame.index)):.3f}  <- the jump toward perfect, by construction")
print(f"Top in-sample coefficients (eyeball only): {', '.join(f'{name} {coef:+.2f}' for name, coef in top)}")
print(f"  ...and when the trap IS included, its coefficient: {coefs_trap[-1]:+.2f} (the model grabs the leak)")
print("Receipts: trap leaves — the honest numbers return. Base rate and both split styles sit next to every score.")

Base rate: 15.9% | coin flip = 0.500 AUC
Grouped CV AUC (client-holdout, out-of-fold):  0.626  | precision@20 on OOF scores: 0.30
Random row split AUC (25% test, stratified):   0.641
  memorization receipt = random - grouped = +0.015
Grouped CV AUC with momentum removed:          0.611
Grouped CV AUC with the trap column added:     1.000  <- the jump toward perfect, by construction
Top in-sample coefficients (eyeball only): imp_early_tot -0.97, momentum +0.97, click_rate_pct -0.95
  ...and when the trap IS included, its coefficient: +51.46 (the model grabs the leak)
Receipts: trap leaves — the honest numbers return. Base rate and both split styles sit next to every score.


## 4. What I excluded and why

Refused fields, one line each — printed by the receipt cell too:

- **GA4 engagement columns** — zero-filled before each client's `ga4_data_start`; learning those zeros teaches onboarding age, not page quality (and only ~4% of March row-days are honest GA4 rows);
- **availability flags (`gsc_data_available`, `ga4_data_available`, `client_has_*`) as features** — they encode client provisioning, not page behavior; filter-only;
- **`fact_content_daily_performance_sample`** — exactly the final month (June 2026): the natural outcome window of any past-to-future label; label logic never develops there;
- **`fact_content_query_90d`** — its fixed 90-day window overlaps my target window, and the query-level grain would change the unit of analysis;
- **product decision columns** (health scores, queue flags) — not shipped in the release (receipt below); if ever handed to me, they are a baseline to beat, never inputs;
- **`client_hash_id` / `content_hash_id` as model inputs** — context and split keys only (privacy plus: nothing to learn by construction).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Receipt: no decision-derived columns hide in the release; the exclusion list stands next to the schema.
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT_MAR}')").df()
cols = schema.column_name.tolist()
suspect = [c for c in cols if any(word in c.lower() for word in ["score", "health", "decision", "queue", "priority", "rank"])]
print(f"March partition columns that look decision-derived: {suspect if suspect else 'none'}")
flags_like = [c for c in cols if c.endswith("_available") or c in {"client_has_gsc", "client_has_ga4"}]
print(f"Availability/provisioning columns found (context, filter-only): {flags_like}")
for name, why in {
    "GA4 engagement columns": "zero-filled before ga4_data_start — onboarding, not page quality",
    "availability flags as features": "provisioning status, not behavior — filters only",
    "the sealed _sample table": "it IS the final month = outcome window territory",
    "fact_content_query_90d": "fixed 90-day window overlaps the target window",
    "product decision columns": "not shipped here; if rebuilt, baseline-to-beat only",
    "client/content hashes as inputs": "context and split keys only (privacy by design)",
}.items():
    print(f"  excluded: {name} — {why}")
print("Privacy: only pseudonymous hashes travel through these notebooks; no client-identifying output is printed anywhere.")

March partition columns that look decision-derived: none
Availability/provisioning columns found (context, filter-only): ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
  excluded: GA4 engagement columns — zero-filled before ga4_data_start — onboarding, not page quality
  excluded: availability flags as features — provisioning status, not behavior — filters only
  excluded: the sealed _sample table — it IS the final month = outcome window territory
  excluded: fact_content_query_90d — fixed 90-day window overlaps the target window
  excluded: product decision columns — not shipped here; if rebuilt, baseline-to-beat only
  excluded: client/content hashes as inputs — context and split keys only (privacy by design)
Privacy: only pseudonymous hashes travel through these notebooks; no client-identifying output is printed anywhere.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.